# Model A — Full Dataset Training
**Intelligent Reading Comprehension and Quiz Generation System**

This notebook trains Model A (Question & Answer Generator/Verifier) on the **full RACE dataset** following the project specification:

- **Verification task**: given (article + question + option), predict if option is correct
- **Generation task**: template-based question generation with ML ranking (SVM / Random Forest)

**Evaluation per supervisor's updated guidance:**
- BLEU, ROUGE, METEOR are the primary generation metrics
- Question-level accuracy is kept as a secondary diagnostic only

**Resume-safe**: every heavy step checks if its output already exists on Drive. If Colab disconnects, just re-run all cells — finished steps will skip.

## 1. Setup — Drive, paths, configuration

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

PROJECT = "/content/drive/MyDrive/race_rc_project"

RAW_DIR        = f"{PROJECT}/data/raw"
PROCESSED_DIR  = f"{PROJECT}/data/processed"
EDA_DIR        = f"{PROJECT}/notebooks/eda_outputs"
MODEL_A_DIR    = f"{PROJECT}/models/model_a"
RESULTS_DIR    = f"{PROJECT}/results"

for path in [RAW_DIR, PROCESSED_DIR, EDA_DIR, MODEL_A_DIR, RESULTS_DIR]:
    os.makedirs(path, exist_ok=True)

print("Project folder ready:", PROJECT)

Mounted at /content/drive
Project folder ready: /content/drive/MyDrive/race_rc_project


In [ ]:
# Imports — consolidated
import re
import gc
import ast
import json
import time
import string
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import joblib
import scipy.sparse as sp
from scipy.sparse import hstack

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix, silhouette_score
)

print("Imports complete.")

Imports complete.


In [ ]:
# Configuration
# Set FULL_MODE = True to run on the full dataset (~100K questions).
# Set FULL_MODE = False to run a quick development pass on 5000 rows.
FULL_MODE = True
#DEV_SAMPLE_SIZE = 5000
RANDOM_STATE = 42

# Suffix for output files. Keeps full-data files separate from any old dev files.
SUFFIX = "_full" if FULL_MODE else "_dev"

RAW_FILE = f"{RAW_DIR}/race.csv"

print("FULL_MODE:", FULL_MODE)
print("Output suffix:", SUFFIX)
print("Raw file path:", RAW_FILE)

FULL_MODE: True
Output suffix: _full
Raw file path: /content/drive/MyDrive/race_rc_project/data/raw/race.csv


In [ ]:
# Verify dataset exists
if not os.path.exists(RAW_FILE):
    raise FileNotFoundError(
        f"Dataset not found at {RAW_FILE}. "
        "Upload your Kaggle CSV and rename it to race.csv"
    )

print("Dataset found.")

Dataset found.


## 2. Load dataset

Loads the full single CSV. If `FULL_MODE = False`, samples 5000 rows for fast iteration.

In [ ]:
df = pd.read_csv(RAW_FILE)

print("Raw shape:", df.shape)
print("Columns:", df.columns.tolist())

if not FULL_MODE:
    sample_size = min(DEV_SAMPLE_SIZE, len(df))
    df = df.sample(n=sample_size, random_state=RANDOM_STATE).reset_index(drop=True)
    print(f"DEV mode: using {sample_size} rows")
else:
    print(f"FULL mode: using all {len(df)} rows")

display(df.head(2))

Raw shape: (87866, 9)
Columns: ['Unnamed: 0', 'id', 'article', 'question', 'A', 'B', 'C', 'D', 'answer']
FULL mode: using all 87866 rows


,Unnamed: 0,id,article,question,A,B,C,D,answer
0,0,middle7348.txt,In the summer between my first year and second...,Before the writer came to the high school summ...,instructor,camper,student,reporter,C
1,1,middle7348.txt,In the summer between my first year and second...,How many times did the writer invite the boy t...,Once,Twice,Three times,Many times,B


In [ ]:
# Standardize column names — Kaggle versions sometimes use lowercase
rename_map = {
    "a": "A", "b": "B", "c": "C", "d": "D",
    "answers": "answer", "label": "answer"
}
df = df.rename(columns=rename_map)

REQUIRED_COLUMNS = ["id", "article", "question", "A", "B", "C", "D", "answer"]
missing = [c for c in REQUIRED_COLUMNS if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df = df[REQUIRED_COLUMNS].copy()
print("All required columns present.")
print("Shape:", df.shape)

All required columns present.
Shape: (87866, 8)


## 3. Quick EDA

Lightweight stats only — saves a summary CSV and a few plots once. If they already exist on Drive they are skipped.

In [ ]:
eda_summary_path = f"{EDA_DIR}/summary_statistics{SUFFIX}.csv"

def word_count(text):
    return len(str(text).split())

if os.path.exists(eda_summary_path):
    print("EDA already done. Loading summary...")
    summary_stats_df = pd.read_csv(eda_summary_path)
else:
    print("Computing EDA...")
    article_wc = df["article"].apply(word_count)
    question_wc = df["question"].apply(word_count)

    summary_stats = {
        "total_rows": int(len(df)),
        "unique_articles": int(df["article"].nunique()),
        "duplicate_rows": int(df.duplicated().sum()),
        "missing_values_total": int(df.isnull().sum().sum()),
        "avg_article_words": float(article_wc.mean()),
        "median_article_words": float(article_wc.median()),
        "max_article_words": int(article_wc.max()),
        "avg_question_words": float(question_wc.mean()),
        "median_question_words": float(question_wc.median()),
    }
    summary_stats_df = pd.DataFrame([summary_stats])
    summary_stats_df.to_csv(eda_summary_path, index=False)
    print("Saved:", eda_summary_path)

display(summary_stats_df)

EDA already done. Loading summary...


,total_rows,unique_articles,duplicate_rows,missing_values_total,avg_article_words,median_article_words,max_article_words,avg_question_words,median_question_words
0,87866,25135,1,14,274.983702,279.0,1162,10.008786,10.0


In [ ]:
# Answer balance check (saved as plot once)
answer_balance_plot = f"{EDA_DIR}/answer_balance{SUFFIX}.png"

if not os.path.exists(answer_balance_plot):
    answer_counts = df["answer"].astype(str).str.upper().value_counts().sort_index()

    plt.figure(figsize=(7, 4))
    plt.bar(answer_counts.index, answer_counts.values, color="steelblue")
    plt.title("Answer Balance (A / B / C / D)")
    plt.xlabel("Correct Answer Label")
    plt.ylabel("Count")
    plt.grid(axis="y", alpha=0.3)
    plt.savefig(answer_balance_plot, bbox_inches="tight", dpi=120)
    plt.show()
    print("Saved:", answer_balance_plot)
else:
    print("Answer balance plot already exists.")
    print(df["answer"].value_counts().sort_index())

Answer balance plot already exists.
answer
A    19146
B    22726
C    23891
D    22103
Name: count, dtype: int64


## 4. Text-cleaning functions

Two cleaning levels:
- `clean_text_basic` — for sentence splitting and display
- `clean_text_no_punct` — for TF-IDF and lexical overlap features

In [ ]:
def clean_text_basic(text):
    text = str(text).lower()
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def clean_text_no_punct(text):
    text = clean_text_basic(text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def split_sentences(article):
    article = str(article)
    sentences = re.split(r"(?<=[.!?])\s+", article)
    return [s.strip() for s in sentences if len(s.strip()) > 0]

print("Cleaning helpers defined.")

Cleaning helpers defined.


## 5. Build cleaned DataFrame with `question_uid`

Critical fix: the original `id` column is NOT unique per question (many questions share the same article id). We add `question_uid` to give every row a unique identifier — needed for grouping options together later.

In [ ]:
clean_df_path = f"{PROCESSED_DIR}/clean_df{SUFFIX}.csv"

def prepare_clean_dataframe(raw_df):
    out = raw_df.copy()
    out = out.dropna(subset=REQUIRED_COLUMNS)

    # Normalize answer labels
    out["answer"] = out["answer"].astype(str).str.strip().str.upper()
    out = out[out["answer"].isin(["A", "B", "C", "D"])]

    # Drop exact duplicates
    out = out.drop_duplicates(
        subset=["article", "question", "A", "B", "C", "D", "answer"]
    )

    # Add cleaned text columns
    text_cols = ["article", "question", "A", "B", "C", "D"]
    for col in text_cols:
        out[col + "_clean"]   = out[col].apply(clean_text_basic)
        out[col + "_nopunct"] = out[col].apply(clean_text_no_punct)

    # Add correct-answer text columns
    out["correct_answer"]         = out.apply(lambda r: r[r["answer"]], axis=1)
    out["correct_answer_clean"]   = out.apply(lambda r: r[r["answer"] + "_clean"], axis=1)
    out["correct_answer_nopunct"] = out.apply(lambda r: r[r["answer"] + "_nopunct"], axis=1)

    # Sentence list (kept as Python list for now; CSV will store as string)
    out["article_sentences"] = out["article_clean"].apply(split_sentences)

    # Reset index, then add unique question_uid
    out = out.reset_index(drop=True)
    out["question_uid"] = ["q_" + str(i) for i in range(len(out))]

    return out

if os.path.exists(clean_df_path):
    print("Clean dataframe already exists. Loading...")
    clean_df = pd.read_csv(clean_df_path)
else:
    print("Building clean dataframe... (this can take a few minutes on full data)")
    clean_df = prepare_clean_dataframe(df)
    clean_df.to_csv(clean_df_path, index=False)
    print("Saved:", clean_df_path)

print("Clean shape:", clean_df.shape)
print("Unique question_uid count:", clean_df["question_uid"].nunique())
display(clean_df[["question_uid", "id", "question", "answer"]].head(3))

Clean dataframe already exists. Loading...
Clean shape: (87851, 25)
Unique question_uid count: 87851


,question_uid,id,question,answer
0,q_0,middle7348.txt,Before the writer came to the high school summ...,C
1,q_1,middle7348.txt,How many times did the writer invite the boy t...,B
2,q_2,middle4305.txt,The bumpkin thought _ .,C


## 6. 80 / 10 / 10 split (stratified by `answer`)

In [ ]:
train_clean_path = f"{PROCESSED_DIR}/train_clean{SUFFIX}.csv"
val_clean_path   = f"{PROCESSED_DIR}/val_clean{SUFFIX}.csv"
test_clean_path  = f"{PROCESSED_DIR}/test_clean{SUFFIX}.csv"

def split_80_10_10(df_in, random_state=42):
    train_df, temp_df = train_test_split(
        df_in, test_size=0.20, random_state=random_state, stratify=df_in["answer"]
    )
    val_df, test_df = train_test_split(
        temp_df, test_size=0.50, random_state=random_state, stratify=temp_df["answer"]
    )
    return (
        train_df.reset_index(drop=True),
        val_df.reset_index(drop=True),
        test_df.reset_index(drop=True),
    )

if all(os.path.exists(p) for p in [train_clean_path, val_clean_path, test_clean_path]):
    print("Clean splits already exist. Loading...")
    train_clean = pd.read_csv(train_clean_path)
    val_clean   = pd.read_csv(val_clean_path)
    test_clean  = pd.read_csv(test_clean_path)
else:
    print("Creating 80/10/10 split...")
    train_clean, val_clean, test_clean = split_80_10_10(clean_df, RANDOM_STATE)
    train_clean.to_csv(train_clean_path, index=False)
    val_clean.to_csv(val_clean_path, index=False)
    test_clean.to_csv(test_clean_path, index=False)
    print("Saved.")

print("Train:", train_clean.shape)
print("Val:  ", val_clean.shape)
print("Test: ", test_clean.shape)
print("\nTrain answer balance:")
print(train_clean["answer"].value_counts(normalize=True).sort_index())

Clean splits already exist. Loading...
Train: (70280, 25)
Val:   (8785, 25)
Test:  (8786, 25)

Train answer balance:
answer
A    0.217886
B    0.258637
C    0.271898
D    0.251579
Name: proportion, dtype: float64


## 7. Build Model A dataset

Each question becomes **4 rows** — one per option (A, B, C, D). Label `is_correct` = 1 only for the correct option. This is what Model A's verifier learns from.

In [ ]:
def build_model_a_dataset(df_in):
    rows = []
    for _, row in df_in.iterrows():
        for option_label in ["A", "B", "C", "D"]:
            rows.append({
                "question_uid": row["question_uid"],
                "id": row["id"],
                "article": row["article_clean"],
                "article_nopunct": row["article_nopunct"],
                "question": row["question_clean"],
                "question_nopunct": row["question_nopunct"],
                "option_label": option_label,
                "option_text": row[option_label + "_clean"],
                "option_text_nopunct": row[option_label + "_nopunct"],
                "correct_answer": row["correct_answer_clean"],
                "correct_answer_nopunct": row["correct_answer_nopunct"],
                "gold_answer_label": row["answer"],
                "is_correct": 1 if option_label == row["answer"] else 0,
            })
    out = pd.DataFrame(rows)
    out["input_text"] = (
        out["article_nopunct"] + " question " +
        out["question_nopunct"] + " option " +
        out["option_text_nopunct"]
    )
    return out

model_a_train_path = f"{PROCESSED_DIR}/model_a_train{SUFFIX}.csv"
model_a_val_path   = f"{PROCESSED_DIR}/model_a_val{SUFFIX}.csv"
model_a_test_path  = f"{PROCESSED_DIR}/model_a_test{SUFFIX}.csv"

if all(os.path.exists(p) for p in [model_a_train_path, model_a_val_path, model_a_test_path]):
    print("Model A datasets already exist. Loading...")
    model_a_train = pd.read_csv(model_a_train_path)
    model_a_val   = pd.read_csv(model_a_val_path)
    model_a_test  = pd.read_csv(model_a_test_path)
else:
    print("Building Model A datasets (4 rows per question)...")
    model_a_train = build_model_a_dataset(train_clean)
    model_a_val   = build_model_a_dataset(val_clean)
    model_a_test  = build_model_a_dataset(test_clean)

    model_a_train.to_csv(model_a_train_path, index=False)
    model_a_val.to_csv(model_a_val_path, index=False)
    model_a_test.to_csv(model_a_test_path, index=False)
    print("Saved.")

print("Model A train rows:", len(model_a_train), " (questions:", len(model_a_train)//4, ")")
print("Model A val rows:  ", len(model_a_val))
print("Model A test rows: ", len(model_a_test))
print("\nLabel distribution (train):")
print(model_a_train["is_correct"].value_counts(normalize=True))

Model A datasets already exist. Loading...
Model A train rows: 281120  (questions: 70280 )
Model A val rows:   35140
Model A test rows:  35144

Label distribution (train):
is_correct
0    0.75
1    0.25
Name: proportion, dtype: float64


## 8. Build Model B dataset (used by question generation)

Even though we are focused on Model A, the **question-generation sub-task** of Model A needs the original question text and full sentence list. We store them in a per-question table for convenience.

In [ ]:
def build_model_b_dataset(df_in):
    rows = []
    for _, row in df_in.iterrows():
        wrong_options = []
        wrong_labels = []
        for label in ["A", "B", "C", "D"]:
            if label != row["answer"]:
                wrong_options.append(row[label + "_clean"])
                wrong_labels.append(label)
        rows.append({
            "question_uid": row["question_uid"],
            "id": row["id"],
            "article": row["article_clean"],
            "article_nopunct": row["article_nopunct"],
            "question": row["question_clean"],
            "question_nopunct": row["question_nopunct"],
            "correct_answer": row["correct_answer_clean"],
            "correct_answer_nopunct": row["correct_answer_nopunct"],
            "gold_answer_label": row["answer"],
            "distractor_1": wrong_options[0],
            "distractor_2": wrong_options[1],
            "distractor_3": wrong_options[2],
            "distractor_label_1": wrong_labels[0],
            "distractor_label_2": wrong_labels[1],
            "distractor_label_3": wrong_labels[2],
            "article_sentences": row["article_sentences"],
        })
    return pd.DataFrame(rows)

model_b_train_path = f"{PROCESSED_DIR}/model_b_train{SUFFIX}.csv"
model_b_val_path   = f"{PROCESSED_DIR}/model_b_val{SUFFIX}.csv"
model_b_test_path  = f"{PROCESSED_DIR}/model_b_test{SUFFIX}.csv"

if all(os.path.exists(p) for p in [model_b_train_path, model_b_val_path, model_b_test_path]):
    print("Model B datasets already exist. Loading...")
    model_b_train = pd.read_csv(model_b_train_path)
    model_b_val   = pd.read_csv(model_b_val_path)
    model_b_test  = pd.read_csv(model_b_test_path)
else:
    print("Building Model B datasets...")
    model_b_train = build_model_b_dataset(train_clean)
    model_b_val   = build_model_b_dataset(val_clean)
    model_b_test  = build_model_b_dataset(test_clean)
    model_b_train.to_csv(model_b_train_path, index=False)
    model_b_val.to_csv(model_b_val_path, index=False)
    model_b_test.to_csv(model_b_test_path, index=False)
    print("Saved.")

# Free original splits and clean_df from RAM (already on disk)
del clean_df, train_clean, val_clean, test_clean
gc.collect()

print("Model B train rows:", len(model_b_train))
print("Model B val rows:  ", len(model_b_val))
print("Model B test rows: ", len(model_b_test))

Model B datasets already exist. Loading...
Model B train rows: 70280
Model B val rows:   8785
Model B test rows:  8786


## 9. TF-IDF features for Model A

The project allows TF-IDF as a vectorizer. Parameters chosen for full-data scale:
- `max_features=50000` — cap vocab so memory stays bounded
- `min_df=5` — drop ultra-rare tokens (more aggressive than dev)
- `max_df=0.95` — drop near-stopwords
- `ngram_range=(1, 2)` — unigrams + bigrams

In [ ]:
tfidf_path = f"{MODEL_A_DIR}/tfidf_vectorizer{SUFFIX}.pkl"

X_train_path = f"{PROCESSED_DIR}/X_model_a_train_tfidf{SUFFIX}.npz"
X_val_path   = f"{PROCESSED_DIR}/X_model_a_val_tfidf{SUFFIX}.npz"
X_test_path  = f"{PROCESSED_DIR}/X_model_a_test_tfidf{SUFFIX}.npz"

y_train_path = f"{PROCESSED_DIR}/y_model_a_train{SUFFIX}.npy"
y_val_path   = f"{PROCESSED_DIR}/y_model_a_val{SUFFIX}.npy"
y_test_path  = f"{PROCESSED_DIR}/y_model_a_test{SUFFIX}.npy"

needed = [tfidf_path, X_train_path, X_val_path, X_test_path,
          y_train_path, y_val_path, y_test_path]

if all(os.path.exists(p) for p in needed):
    print("TF-IDF artefacts already exist. Loading...")
    tfidf         = joblib.load(tfidf_path)
    X_train_tfidf = sp.load_npz(X_train_path)
    X_val_tfidf   = sp.load_npz(X_val_path)
    X_test_tfidf  = sp.load_npz(X_test_path)
    y_train = np.load(y_train_path)
    y_val   = np.load(y_val_path)
    y_test  = np.load(y_test_path)
else:
    print("Fitting TF-IDF on training input_text...")
    tfidf = TfidfVectorizer(
        max_features=50000,
        ngram_range=(1, 2),
        min_df=5,
        max_df=0.95,
    )
    X_train_tfidf = tfidf.fit_transform(model_a_train["input_text"])
    X_val_tfidf   = tfidf.transform(model_a_val["input_text"])
    X_test_tfidf  = tfidf.transform(model_a_test["input_text"])

    y_train = model_a_train["is_correct"].values
    y_val   = model_a_val["is_correct"].values
    y_test  = model_a_test["is_correct"].values

    joblib.dump(tfidf, tfidf_path)
    sp.save_npz(X_train_path, X_train_tfidf)
    sp.save_npz(X_val_path, X_val_tfidf)
    sp.save_npz(X_test_path, X_test_tfidf)
    np.save(y_train_path, y_train)
    np.save(y_val_path, y_val)
    np.save(y_test_path, y_test)
    print("Saved TF-IDF + feature matrices.")

print("X_train_tfidf:", X_train_tfidf.shape)
print("X_val_tfidf:  ", X_val_tfidf.shape)
print("X_test_tfidf: ", X_test_tfidf.shape)
print("y_train:", y_train.shape, "  positive rate:", y_train.mean())

TF-IDF artefacts already exist. Loading...
X_train_tfidf: (281120, 50000)
X_val_tfidf:   (35140, 50000)
X_test_tfidf:  (35144, 50000)
y_train: (281120,)   positive rate: 0.25


## 10. Cosine + lexical features

For each `(article, question, option)` triple we compute:
- 3 cosine similarities (article-option, question-option, article-question)
- 11 lexical features (lengths, word-overlap counts, overlap ratios)

These are saved to CSV — they are small and cheap.

In [ ]:
def add_cosine_features(df_in, vectorizer):
    df_out = df_in.copy()
    # TF-IDF vectors are L2-normalized, so dot product = cosine similarity
    article_vec  = vectorizer.transform(df_out["article_nopunct"].astype(str))
    question_vec = vectorizer.transform(df_out["question_nopunct"].astype(str))
    option_vec   = vectorizer.transform(df_out["option_text_nopunct"].astype(str))

    df_out["cos_article_option"]   = np.asarray(article_vec.multiply(option_vec).sum(axis=1)).ravel()
    df_out["cos_question_option"]  = np.asarray(question_vec.multiply(option_vec).sum(axis=1)).ravel()
    df_out["cos_article_question"] = np.asarray(article_vec.multiply(question_vec).sum(axis=1)).ravel()
    return df_out

def word_overlap_count(t1, t2):
    return len(set(str(t1).split()) & set(str(t2).split()))

def word_overlap_ratio(t1, t2):
    s1, s2 = set(str(t1).split()), set(str(t2).split())
    return (len(s1 & s2) / len(s2)) if len(s2) else 0.0

def add_lexical_features(df_in):
    df_out = df_in.copy()
    df_out["article_len"]  = df_out["article_nopunct"].apply(lambda x: len(str(x).split()))
    df_out["question_len"] = df_out["question_nopunct"].apply(lambda x: len(str(x).split()))
    df_out["option_len"]   = df_out["option_text_nopunct"].apply(lambda x: len(str(x).split()))

    df_out["question_option_overlap"]  = df_out.apply(
        lambda r: word_overlap_count(r["question_nopunct"], r["option_text_nopunct"]), axis=1)
    df_out["article_option_overlap"]   = df_out.apply(
        lambda r: word_overlap_count(r["article_nopunct"], r["option_text_nopunct"]), axis=1)
    df_out["article_question_overlap"] = df_out.apply(
        lambda r: word_overlap_count(r["article_nopunct"], r["question_nopunct"]), axis=1)
    df_out["question_option_overlap_ratio"] = df_out.apply(
        lambda r: word_overlap_ratio(r["question_nopunct"], r["option_text_nopunct"]), axis=1)
    df_out["article_option_overlap_ratio"]  = df_out.apply(
        lambda r: word_overlap_ratio(r["article_nopunct"], r["option_text_nopunct"]), axis=1)
    return df_out

features_train_path = f"{PROCESSED_DIR}/model_a_train_features{SUFFIX}.csv"
features_val_path   = f"{PROCESSED_DIR}/model_a_val_features{SUFFIX}.csv"
features_test_path  = f"{PROCESSED_DIR}/model_a_test_features{SUFFIX}.csv"

if all(os.path.exists(p) for p in [features_train_path, features_val_path, features_test_path]):
    print("Cosine + lexical features already exist. Loading...")
    model_a_train_features = pd.read_csv(features_train_path)
    model_a_val_features   = pd.read_csv(features_val_path)
    model_a_test_features  = pd.read_csv(features_test_path)
else:
    print("Computing cosine + lexical features...")
    print("  train...");  tmp_train = add_cosine_features(model_a_train, tfidf)
    print("  val...");    tmp_val   = add_cosine_features(model_a_val,   tfidf)
    print("  test...");   tmp_test  = add_cosine_features(model_a_test,  tfidf)

    model_a_train_features = add_lexical_features(tmp_train)
    model_a_val_features   = add_lexical_features(tmp_val)
    model_a_test_features  = add_lexical_features(tmp_test)
    del tmp_train, tmp_val, tmp_test; gc.collect()

    model_a_train_features.to_csv(features_train_path, index=False)
    model_a_val_features.to_csv(features_val_path, index=False)
    model_a_test_features.to_csv(features_test_path, index=False)
    print("Saved.")

# Free the per-row datasets — we have what we need in *_features now
del model_a_train, model_a_val, model_a_test
gc.collect()

print("Train features:", model_a_train_features.shape)
print("Val features:  ", model_a_val_features.shape)
print("Test features: ", model_a_test_features.shape)

Cosine + lexical features already exist. Loading...
Train features: (281120, 25)
Val features:   (35140, 25)
Test features:  (35144, 25)


## 11. Combined feature matrix (TF-IDF + scaled numeric)

In [ ]:
NUMERIC_COLS = [
    "cos_article_option", "cos_question_option", "cos_article_question",
    "article_len", "question_len", "option_len",
    "question_option_overlap", "article_option_overlap", "article_question_overlap",
    "question_option_overlap_ratio", "article_option_overlap_ratio",
]

scaler_path = f"{MODEL_A_DIR}/numeric_scaler{SUFFIX}.pkl"

X_train_num_raw = model_a_train_features[NUMERIC_COLS].fillna(0).values
X_val_num_raw   = model_a_val_features[NUMERIC_COLS].fillna(0).values
X_test_num_raw  = model_a_test_features[NUMERIC_COLS].fillna(0).values

if os.path.exists(scaler_path):
    scaler = joblib.load(scaler_path)
    print("Loaded existing numeric scaler.")
else:
    scaler = StandardScaler()
    scaler.fit(X_train_num_raw)
    joblib.dump(scaler, scaler_path)
    print("Fitted and saved numeric scaler.")

X_train_num = scaler.transform(X_train_num_raw)
X_val_num   = scaler.transform(X_val_num_raw)
X_test_num  = scaler.transform(X_test_num_raw)

# Combined sparse matrix
X_train_combined_path = f"{PROCESSED_DIR}/X_model_a_train_combined{SUFFIX}.npz"
X_val_combined_path   = f"{PROCESSED_DIR}/X_model_a_val_combined{SUFFIX}.npz"
X_test_combined_path  = f"{PROCESSED_DIR}/X_model_a_test_combined{SUFFIX}.npz"

if all(os.path.exists(p) for p in [X_train_combined_path, X_val_combined_path, X_test_combined_path]):
    print("Combined feature matrices already exist. Loading...")
    X_train_combined = sp.load_npz(X_train_combined_path)
    X_val_combined   = sp.load_npz(X_val_combined_path)
    X_test_combined  = sp.load_npz(X_test_combined_path)
else:
    print("Building combined sparse matrices...")
    X_train_combined = hstack([X_train_tfidf, sp.csr_matrix(X_train_num)]).tocsr()
    X_val_combined   = hstack([X_val_tfidf,   sp.csr_matrix(X_val_num)]).tocsr()
    X_test_combined  = hstack([X_test_tfidf,  sp.csr_matrix(X_test_num)]).tocsr()
    sp.save_npz(X_train_combined_path, X_train_combined)
    sp.save_npz(X_val_combined_path,   X_val_combined)
    sp.save_npz(X_test_combined_path,  X_test_combined)
    print("Saved.")

print("X_train_combined:", X_train_combined.shape)
print("X_val_combined:  ", X_val_combined.shape)
print("X_test_combined: ", X_test_combined.shape)

Loaded existing numeric scaler.
Combined feature matrices already exist. Loading...
X_train_combined: (281120, 50011)
X_val_combined:   (35140, 50011)
X_test_combined:  (35144, 50011)


## 12. BLEU / ROUGE / METEOR setup

Per supervisor's update: these are the **primary evaluation metrics** for both verification (selected answer text vs reference) and question generation.

In [ ]:
!pip -q install rouge-score nltk

import nltk
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer

smooth = SmoothingFunction().method1
rouge  = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

def normalize_text(text):
    return re.sub(r"\s+", " ", str(text).lower().strip())

def simple_tokenize(text):
    return normalize_text(text).split()

def compute_generation_metrics(reference, generated):
    ref = normalize_text(reference)
    gen = normalize_text(generated)
    ref_tok = simple_tokenize(ref)
    gen_tok = simple_tokenize(gen)
    if len(gen_tok) == 0 or len(ref_tok) == 0:
        return {k: 0.0 for k in ["bleu_1","bleu_2","bleu_4","rouge1_f1","rouge2_f1","rougeL_f1","meteor"]}
    bleu_1 = sentence_bleu([ref_tok], gen_tok, weights=(1,0,0,0),       smoothing_function=smooth)
    bleu_2 = sentence_bleu([ref_tok], gen_tok, weights=(0.5,0.5,0,0),   smoothing_function=smooth)
    bleu_4 = sentence_bleu([ref_tok], gen_tok, weights=(0.25,)*4,       smoothing_function=smooth)
    rs = rouge.score(ref, gen)
    try:
        m = meteor_score([ref_tok], gen_tok)
    except Exception:
        m = 0.0
    return {
        "bleu_1": float(bleu_1), "bleu_2": float(bleu_2), "bleu_4": float(bleu_4),
        "rouge1_f1": float(rs["rouge1"].fmeasure),
        "rouge2_f1": float(rs["rouge2"].fmeasure),
        "rougeL_f1": float(rs["rougeL"].fmeasure),
        "meteor": float(m),
    }

def evaluate_generation_dataframe(df_in, ref_col, gen_col):
    rows = []
    for _, r in df_in.iterrows():
        rows.append(compute_generation_metrics(r[ref_col], r[gen_col]))
    metrics_df = pd.DataFrame(rows)
    return metrics_df.mean().to_dict(), metrics_df

print("BLEU / ROUGE / METEOR ready.")

  Preparing metadata (setup.py) ... done
BLEU / ROUGE / METEOR ready.


## 13. Training helpers

`train_or_load_model` saves/loads from disk so a dropped Colab session never re-trains a finished model.

In [ ]:
MODEL_A_RESULTS_PATH    = f"{RESULTS_DIR}/model_a_results{SUFFIX}.json"
MODEL_A_COMPARISON_CSV  = f"{RESULTS_DIR}/model_a_comparison{SUFFIX}.csv"

def load_results():
    if os.path.exists(MODEL_A_RESULTS_PATH):
        with open(MODEL_A_RESULTS_PATH) as f:
            return json.load(f)
    return {}

def save_results(results):
    with open(MODEL_A_RESULTS_PATH, "w") as f:
        json.dump(results, f, indent=2)

def train_or_load_model(model_path, builder, X, y, name):
    if os.path.exists(model_path):
        print(f"Found saved model: {name}. Loading...")
        return joblib.load(model_path), 0.0
    print(f"Training: {name} ...")
    model = builder()
    t0 = time.time()
    model.fit(X, y)
    t = time.time() - t0
    joblib.dump(model, model_path)
    print(f"  trained in {t:.1f}s — saved")
    return model, t

def diagnostic_classification_report(model, X, y, name, split):
    y_pred = model.predict(X)
    return {
        "model_name": name, "split": split,
        "diag_accuracy": float(accuracy_score(y, y_pred)),
        "diag_f1_macro": float(f1_score(y, y_pred, average="macro", zero_division=0)),
        "diag_f1_correct": float(f1_score(y, y_pred, pos_label=1, zero_division=0)),
        "confusion_matrix": confusion_matrix(y, y_pred).tolist(),
    }

print("Training helpers ready.")

Training helpers ready.


## 14. Train Logistic Regression + TF-IDF

In [ ]:
results = load_results()
lr_path = f"{MODEL_A_DIR}/logistic_regression_tfidf{SUFFIX}.pkl"

def build_lr_tfidf():
    return LogisticRegression(
        max_iter=2000, class_weight="balanced",
        solver="liblinear", random_state=42
    )

lr_tfidf, lr_t = train_or_load_model(lr_path, build_lr_tfidf,
                                     X_train_tfidf, y_train,
                                     "Logistic Regression + TF-IDF")
diag = diagnostic_classification_report(lr_tfidf, X_val_tfidf, y_val,
                                        "Logistic Regression + TF-IDF", "validation")
diag["features"] = "TF-IDF"; diag["training_time_seconds"] = lr_t
results["logistic_regression_tfidf"] = diag
save_results(results)
print("Diagnostic option-level F1 (validation):", diag["diag_f1_macro"])

Found saved model: Logistic Regression + TF-IDF. Loading...
Diagnostic option-level F1 (validation): 0.4945439693350323


## 15. Train Linear SVM + TF-IDF

In [ ]:
results = load_results()
svm_path = f"{MODEL_A_DIR}/linear_svm_tfidf{SUFFIX}.pkl"

def build_svm_tfidf():
    # dual=False is faster when n_samples >> n_features (our case)
    return LinearSVC(class_weight="balanced", random_state=42,
                     max_iter=5000, dual=False)

svm_tfidf, svm_t = train_or_load_model(svm_path, build_svm_tfidf,
                                       X_train_tfidf, y_train,
                                       "Linear SVM + TF-IDF")
diag = diagnostic_classification_report(svm_tfidf, X_val_tfidf, y_val,
                                        "Linear SVM + TF-IDF", "validation")
diag["features"] = "TF-IDF"; diag["training_time_seconds"] = svm_t
results["linear_svm_tfidf"] = diag
save_results(results)
print("Diagnostic option-level F1 (validation):", diag["diag_f1_macro"])

Found saved model: Linear SVM + TF-IDF. Loading...
Diagnostic option-level F1 (validation): 0.5031387067080324


## 16. Train Naive Bayes + TF-IDF

In [ ]:
results = load_results()
nb_path = f"{MODEL_A_DIR}/naive_bayes_tfidf{SUFFIX}.pkl"

def build_nb_tfidf():
    return MultinomialNB(alpha=1.0)

nb_tfidf, nb_t = train_or_load_model(nb_path, build_nb_tfidf,
                                     X_train_tfidf, y_train,
                                     "Naive Bayes + TF-IDF")
diag = diagnostic_classification_report(nb_tfidf, X_val_tfidf, y_val,
                                        "Naive Bayes + TF-IDF", "validation")
diag["features"] = "TF-IDF"; diag["training_time_seconds"] = nb_t
results["naive_bayes_tfidf"] = diag
save_results(results)
print("Diagnostic option-level F1 (validation):", diag["diag_f1_macro"])

Found saved model: Naive Bayes + TF-IDF. Loading...
Diagnostic option-level F1 (validation): 0.42857142857142855


## 17. Train Random Forest + numeric features

In [ ]:
results = load_results()
rf_path = f"{MODEL_A_DIR}/random_forest_numeric{SUFFIX}.pkl"

def build_rf_numeric():
    return RandomForestClassifier(
        n_estimators=200, max_depth=None,
        class_weight="balanced", random_state=42, n_jobs=-1
    )

rf_num, rf_t = train_or_load_model(rf_path, build_rf_numeric,
                                   X_train_num, y_train,
                                   "Random Forest + Numeric")
diag = diagnostic_classification_report(rf_num, X_val_num, y_val,
                                        "Random Forest + Numeric", "validation")
diag["features"] = "cosine + lexical"; diag["training_time_seconds"] = rf_t
results["random_forest_numeric"] = diag
save_results(results)
print("Diagnostic option-level F1 (validation):", diag["diag_f1_macro"])

Found saved model: Random Forest + Numeric. Loading...
Diagnostic option-level F1 (validation): 0.46324926106882325


## 18. Train Logistic Regression + combined features

In [ ]:
results = load_results()
lr_comb_path = f"{MODEL_A_DIR}/logistic_regression_combined{SUFFIX}.pkl"

def build_lr_combined():
    return LogisticRegression(
        max_iter=2000, class_weight="balanced",
        solver="liblinear", random_state=42
    )

lr_comb, t = train_or_load_model(lr_comb_path, build_lr_combined,
                                 X_train_combined, y_train,
                                 "Logistic Regression + Combined")
diag = diagnostic_classification_report(lr_comb, X_val_combined, y_val,
                                        "Logistic Regression + Combined", "validation")
diag["features"] = "TF-IDF + cosine + lexical"; diag["training_time_seconds"] = t
results["logistic_regression_combined"] = diag
save_results(results)
print("Diagnostic option-level F1 (validation):", diag["diag_f1_macro"])

Found saved model: Logistic Regression + Combined. Loading...
Diagnostic option-level F1 (validation): 0.5081480312665339


## 19. Train Linear SVM + combined features

In [ ]:
results = load_results()
svm_comb_path = f"{MODEL_A_DIR}/linear_svm_combined{SUFFIX}.pkl"

def build_svm_combined():
    return LinearSVC(class_weight="balanced", random_state=42,
                     max_iter=5000, dual=False)

svm_comb, t = train_or_load_model(svm_comb_path, build_svm_combined,
                                  X_train_combined, y_train,
                                  "Linear SVM + Combined")
diag = diagnostic_classification_report(svm_comb, X_val_combined, y_val,
                                        "Linear SVM + Combined", "validation")
diag["features"] = "TF-IDF + cosine + lexical"; diag["training_time_seconds"] = t
results["linear_svm_combined"] = diag
save_results(results)
print("Diagnostic option-level F1 (validation):", diag["diag_f1_macro"])

Found saved model: Linear SVM + Combined. Loading...
Diagnostic option-level F1 (validation): 0.5176041965197904


## 20. Train Hard Voting Ensemble (LR + SVM + NB)

In [ ]:
results = load_results()
ens_path = f"{MODEL_A_DIR}/hard_voting_ensemble_tfidf{SUFFIX}.pkl"

def build_voting():
    return VotingClassifier(
        estimators=[
            ("lr",  LogisticRegression(max_iter=2000, class_weight="balanced",
                                       solver="liblinear", random_state=42)),
            ("svm", LinearSVC(class_weight="balanced", random_state=42,
                              max_iter=5000, dual=False)),
            ("nb",  MultinomialNB(alpha=1.0)),
        ],
        voting="hard",
    )

ens, t = train_or_load_model(ens_path, build_voting,
                             X_train_tfidf, y_train,
                             "Hard Voting Ensemble + TF-IDF")
diag = diagnostic_classification_report(ens, X_val_tfidf, y_val,
                                        "Hard Voting Ensemble + TF-IDF", "validation")
diag["features"] = "TF-IDF"; diag["training_time_seconds"] = t
diag["ensemble_type"] = "hard voting"
results["hard_voting_ensemble_tfidf"] = diag
save_results(results)
print("Diagnostic option-level F1 (validation):", diag["diag_f1_macro"])

Found saved model: Hard Voting Ensemble + TF-IDF. Loading...
Diagnostic option-level F1 (validation): 0.5150565655474134


## 21. Question-level evaluation + BLEU/ROUGE/METEOR

This is the **headline evaluation per the supervisor's updated guidance**.

For every question, score all 4 options and pick the highest-scoring one. The selected option's text is treated as the *generated answer*. We measure how close it is to the reference (gold) answer text using BLEU, ROUGE, and METEOR. Question-level accuracy is reported alongside as a diagnostic.

In [ ]:
def get_model_scores(model, X):
    """Return one positive-class score per row."""
    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(X)
        classes = list(model.classes_)
        idx = classes.index(1) if 1 in classes else 1
        return proba[:, idx]
    if hasattr(model, "decision_function"):
        s = model.decision_function(X)
        return s if s.ndim == 1 else s[:, 1]
    return model.predict(X)

def generate_selected_answers(model, X, df_in):
    tmp = df_in.copy().reset_index(drop=True)
    tmp["score"] = get_model_scores(model, X)
    rows = []
    for q_uid, group in tmp.groupby("question_uid"):
        best = group.loc[group["score"].idxmax()]
        gold = group[group["is_correct"] == 1].iloc[0]
        rows.append({
            "question_uid": q_uid,
            "id": best["id"],
            "reference_question": gold["question"],
            "predicted_label": best["option_label"],
            "gold_label": gold["option_label"],
            "is_correct_question": int(best["option_label"] == gold["option_label"]),
            "generated_answer": best["option_text"],
            "reference_answer": gold["option_text"],
        })
    return pd.DataFrame(rows)

GEN_EVAL_DIR = f"{RESULTS_DIR}/generation_evaluation{SUFFIX}"
os.makedirs(GEN_EVAL_DIR, exist_ok=True)
print("Selection helpers ready. Eval dir:", GEN_EVAL_DIR)

Selection helpers ready. Eval dir: /content/drive/MyDrive/race_rc_project/results/generation_evaluation_full


In [ ]:
# Registry of all trained Model A models
model_registry = {
    "logistic_regression_tfidf": dict(
        model_name="Logistic Regression + TF-IDF",
        path=f"{MODEL_A_DIR}/logistic_regression_tfidf{SUFFIX}.pkl",
        X_val=X_val_tfidf, X_test=X_test_tfidf),
    "linear_svm_tfidf": dict(
        model_name="Linear SVM + TF-IDF",
        path=f"{MODEL_A_DIR}/linear_svm_tfidf{SUFFIX}.pkl",
        X_val=X_val_tfidf, X_test=X_test_tfidf),
    "naive_bayes_tfidf": dict(
        model_name="Naive Bayes + TF-IDF",
        path=f"{MODEL_A_DIR}/naive_bayes_tfidf{SUFFIX}.pkl",
        X_val=X_val_tfidf, X_test=X_test_tfidf),
    "random_forest_numeric": dict(
        model_name="Random Forest + Numeric",
        path=f"{MODEL_A_DIR}/random_forest_numeric{SUFFIX}.pkl",
        X_val=X_val_num, X_test=X_test_num),
    "logistic_regression_combined": dict(
        model_name="Logistic Regression + Combined",
        path=f"{MODEL_A_DIR}/logistic_regression_combined{SUFFIX}.pkl",
        X_val=X_val_combined, X_test=X_test_combined),
    "linear_svm_combined": dict(
        model_name="Linear SVM + Combined",
        path=f"{MODEL_A_DIR}/linear_svm_combined{SUFFIX}.pkl",
        X_val=X_val_combined, X_test=X_test_combined),
    "hard_voting_ensemble_tfidf": dict(
        model_name="Hard Voting Ensemble + TF-IDF",
        path=f"{MODEL_A_DIR}/hard_voting_ensemble_tfidf{SUFFIX}.pkl",
        X_val=X_val_tfidf, X_test=X_test_tfidf),
}

for k, v in model_registry.items():
    print(f"  {k:35s} {'OK' if os.path.exists(v['path']) else 'MISSING'}")

  logistic_regression_tfidf           OK
  linear_svm_tfidf                    OK
  naive_bayes_tfidf                   OK
  random_forest_numeric               OK
  logistic_regression_combined        OK
  linear_svm_combined                 OK
  hard_voting_ensemble_tfidf          OK


In [ ]:
# Evaluate every model on validation: question-level accuracy + BLEU/ROUGE/METEOR
val_results = []
val_pred_dir = f"{GEN_EVAL_DIR}/answer_predictions_validation"
os.makedirs(val_pred_dir, exist_ok=True)

for key, info in model_registry.items():
    print(f"\n>>> {info['model_name']}")
    model = joblib.load(info["path"])
    pred_df = generate_selected_answers(model, info["X_val"], model_a_val_features)

    avg, row_metrics = evaluate_generation_dataframe(
        pred_df, ref_col="reference_answer", gen_col="generated_answer"
    )
    out_df = pd.concat([pred_df, row_metrics], axis=1)
    out_df.to_csv(f"{val_pred_dir}/{key}_val_predictions.csv", index=False)

    q_acc = pred_df["is_correct_question"].mean()
    val_results.append({
        "model_key": key,
        "model_name": info["model_name"],
        "split": "validation",
        "total_questions": int(len(pred_df)),
        "correct_questions": int(pred_df["is_correct_question"].sum()),
        "question_level_accuracy": float(q_acc),
        **avg,
    })
    print(f"  q-acc={q_acc:.4f}  bleu_1={avg['bleu_1']:.4f}  "
          f"rougeL={avg['rougeL_f1']:.4f}  meteor={avg['meteor']:.4f}")

val_results_df = pd.DataFrame(val_results).sort_values(
    by=["meteor", "rougeL_f1", "bleu_1", "question_level_accuracy"],
    ascending=False,
).reset_index(drop=True)

display(val_results_df)
val_results_df.to_csv(f"{GEN_EVAL_DIR}/model_a_validation_metrics.csv", index=False)
print("\nSaved validation metrics.")


>>> Logistic Regression + TF-IDF
  q-acc=0.3342  bleu_1=0.4471  rougeL=0.4666  meteor=0.4052

>>> Linear SVM + TF-IDF
  q-acc=0.3317  bleu_1=0.4466  rougeL=0.4657  meteor=0.4049

>>> Naive Bayes + TF-IDF
  q-acc=0.3040  bleu_1=0.4223  rougeL=0.4439  meteor=0.3807

>>> Random Forest + Numeric
  q-acc=0.3142  bleu_1=0.4291  rougeL=0.4498  meteor=0.3889

>>> Logistic Regression + Combined
  q-acc=0.3620  bleu_1=0.4694  rougeL=0.4872  meteor=0.4283

>>> Linear SVM + Combined
  q-acc=0.3732  bleu_1=0.4802  rougeL=0.4971  meteor=0.4395

>>> Hard Voting Ensemble + TF-IDF
  q-acc=0.2515  bleu_1=0.3799  rougeL=0.4035  meteor=0.3401


,model_key,model_name,split,total_questions,correct_questions,question_level_accuracy,bleu_1,bleu_2,bleu_4,rouge1_f1,rouge2_f1,rougeL_f1,meteor
0,linear_svm_combined,Linear SVM + Combined,validation,8785,3279,0.373250,0.480183,0.402585,0.331088,0.503811,0.374309,0.497137,0.439535
1,logistic_regression_combined,Logistic Regression + Combined,validation,8785,3180,0.361981,0.469449,0.390702,0.319864,0.493651,0.362213,0.487152,0.428319
2,logistic_regression_tfidf,Logistic Regression + TF-IDF,validation,8785,2936,0.334206,0.447075,0.369815,0.305456,0.473087,0.341011,0.466623,0.405155
3,linear_svm_tfidf,Linear SVM + TF-IDF,validation,8785,2914,0.331702,0.446573,0.369078,0.305266,0.472316,0.340215,0.465688,0.404887
4,random_forest_numeric,Random Forest + Numeric,validation,8785,2760,0.314172,0.429136,0.351840,0.287907,0.456905,0.324295,0.449816,0.388905
5,naive_bayes_tfidf,Naive Bayes + TF-IDF,validation,8785,2671,0.304041,0.422279,0.344006,0.277385,0.450307,0.316003,0.443928,0.380717
6,hard_voting_ensemble_tfidf,Hard Voting Ensemble + TF-IDF,validation,8785,2209,0.251451,0.379915,0.301719,0.239591,0.410875,0.273699,0.403469,0.340114



Saved validation metrics.


In [ ]:
# Pick best model by METEOR (most lenient generation metric for short answers),
# then evaluate it on the held-out test set.
best_key  = val_results_df.iloc[0]["model_key"]
best_name = val_results_df.iloc[0]["model_name"]
print("Best validation model:", best_key, "->", best_name)

best_info  = model_registry[best_key]
best_model = joblib.load(best_info["path"])

test_pred_df = generate_selected_answers(best_model, best_info["X_test"], model_a_test_features)
test_avg, test_row_metrics = evaluate_generation_dataframe(
    test_pred_df, ref_col="reference_answer", gen_col="generated_answer"
)
test_out = pd.concat([test_pred_df, test_row_metrics], axis=1)
test_out.to_csv(f"{GEN_EVAL_DIR}/{best_key}_test_predictions.csv", index=False)

test_summary = {
    "best_model_key": best_key,
    "best_model_name": best_name,
    "split": "test",
    "selection_metric": "validation METEOR",
    "total_questions": int(len(test_pred_df)),
    "correct_questions": int(test_pred_df["is_correct_question"].sum()),
    "question_level_accuracy": float(test_pred_df["is_correct_question"].mean()),
    **test_avg,
}
with open(f"{GEN_EVAL_DIR}/best_answer_generation_test_metrics.json", "w") as f:
    json.dump(test_summary, f, indent=2)

print("\nTEST results:")
print(json.dumps(test_summary, indent=2))

Best validation model: linear_svm_combined -> Linear SVM + Combined

TEST results:
{
  "best_model_key": "linear_svm_combined",
  "best_model_name": "Linear SVM + Combined",
  "split": "test",
  "selection_metric": "validation METEOR",
  "total_questions": 8786,
  "correct_questions": 3279,
  "question_level_accuracy": 0.37320737536990667,
  "bleu_1": 0.4806499170055931,
  "bleu_2": 0.40349749974411425,
  "bleu_4": 0.332450130701045,
  "rouge1_f1": 0.5043186161527498,
  "rouge2_f1": 0.3746278812494907,
  "rougeL_f1": 0.49713941791108013,
  "meteor": 0.44003015117581495
}


## 22. Unsupervised — MiniBatch K-Means

Unsupervised baseline (worth 20% of the project marks). We cluster Model A rows in TF-IDF space and report **silhouette** + **purity** against the `is_correct` label.

In [ ]:
results = load_results()
kmeans_path = f"{MODEL_A_DIR}/kmeans_tfidf{SUFFIX}.pkl"

if os.path.exists(kmeans_path):
    print("K-Means model already exists. Loading...")
    kmeans_model = joblib.load(kmeans_path)
    train_t = 0.0
else:
    print("Training MiniBatchKMeans on TF-IDF training matrix...")
    kmeans_model = MiniBatchKMeans(
        n_clusters=2, random_state=42,
        batch_size=4096, n_init=10
    )
    t0 = time.time()
    kmeans_model.fit(X_train_tfidf)
    train_t = time.time() - t0
    joblib.dump(kmeans_model, kmeans_path)

train_clusters = kmeans_model.predict(X_train_tfidf)

def clustering_purity(y_true, clusters):
    total = 0
    for c in np.unique(clusters):
        labels = y_true[clusters == c]
        if len(labels) == 0: continue
        total += np.bincount(labels).max()
    return total / len(y_true)

purity = clustering_purity(y_train, train_clusters)

# Silhouette is O(n^2) — sample to keep memory + time bounded
sample_n = min(5000, X_train_tfidf.shape[0])
rng = np.random.RandomState(42)
sample_idx = rng.choice(X_train_tfidf.shape[0], size=sample_n, replace=False)
sil = silhouette_score(X_train_tfidf[sample_idx], train_clusters[sample_idx])

results["kmeans_tfidf"] = {
    "model_name": "MiniBatch K-Means + TF-IDF",
    "task": "unsupervised_clustering",
    "features": "TF-IDF", "n_clusters": 2,
    "purity": float(purity),
    "silhouette_score_sample": float(sil),
    "silhouette_sample_size": int(sample_n),
    "training_time_seconds": float(train_t),
}
save_results(results)
print(f"K-Means: purity={purity:.4f}  silhouette(sample)={sil:.4f}  train_time={train_t:.1f}s")

K-Means model already exists. Loading...
K-Means: purity=0.7500  silhouette(sample)=0.0062  train_time=0.0s


## 23. Question generation — template baseline (simple)

A naive template generator that uses heuristic question type and produces one generic question per passage. This is the **baseline** that the ML-ranked version below will be compared against.

In [ ]:
def parse_sentence_list(value):
    if isinstance(value, list):
        return value
    try:
        parsed = ast.literal_eval(value)
        if isinstance(parsed, list):
            return parsed
    except Exception:
        pass
    return [str(value)]

def baseline_answer_type(answer):
    a = str(answer).strip()
    if any(c.isdigit() for c in a):
        return "when"
    words = a.split()
    if len(words) >= 2 and all(w[:1].isalpha() for w in words):
        return "who"
    return "what"

def baseline_find_answer_sentence(sentences, answer):
    a_norm = normalize_text(answer)
    for s in sentences:
        if a_norm in normalize_text(s):
            return s
    return sentences[0] if sentences else ""

def baseline_generate(article, correct_answer, article_sentences):
    sents = parse_sentence_list(article_sentences)
    src   = baseline_find_answer_sentence(sents, correct_answer)
    qt    = baseline_answer_type(correct_answer)
    if qt == "when":
        gen = "When did the event mentioned in the passage happen?"
    elif qt == "who":
        gen = "Who is mentioned in the passage?"
    else:
        gen = "What information is given in the passage?"
    return gen, src

def run_baseline(df_in):
    rows = []
    for _, r in df_in.iterrows():
        gen, src = baseline_generate(r["article"], r["correct_answer"], r["article_sentences"])
        rows.append({
            "question_uid": r["question_uid"], "id": r["id"],
            "reference_question": r["question"],
            "generated_question": gen,
            "correct_answer": r["correct_answer"],
            "source_sentence": src,
        })
    return pd.DataFrame(rows)

baseline_val_path  = f"{GEN_EVAL_DIR}/template_baseline_validation.csv"
baseline_test_path = f"{GEN_EVAL_DIR}/template_baseline_test.csv"

if os.path.exists(baseline_val_path) and os.path.exists(baseline_test_path):
    print("Baseline question generation already exists. Loading...")
    baseline_val_df  = pd.read_csv(baseline_val_path)
    baseline_test_df = pd.read_csv(baseline_test_path)
else:
    print("Running baseline question generation...")
    baseline_val_df  = run_baseline(model_b_val)
    baseline_test_df = run_baseline(model_b_test)
    baseline_val_df.to_csv(baseline_val_path, index=False)
    baseline_test_df.to_csv(baseline_test_path, index=False)

baseline_val_avg,  _ = evaluate_generation_dataframe(baseline_val_df,  "reference_question", "generated_question")
baseline_test_avg, _ = evaluate_generation_dataframe(baseline_test_df, "reference_question", "generated_question")

baseline_summary = {
    "method": "template_baseline",
    "validation": baseline_val_avg,
    "test": baseline_test_avg,
}
with open(f"{GEN_EVAL_DIR}/template_baseline_summary.json", "w") as f:
    json.dump(baseline_summary, f, indent=2)

print("Baseline test:", json.dumps(baseline_test_avg, indent=2))

Baseline question generation already exists. Loading...
Baseline test: {
  "bleu_1": 0.11850798209807321,
  "bleu_2": 0.04717004857128457,
  "bleu_4": 0.023834343677906506,
  "rouge1_f1": 0.19057764449679326,
  "rouge2_f1": 0.04051019864692474,
  "rougeL_f1": 0.16999457913037724,
  "meteor": 0.09138999190627925
}


## 24. Question generation — Template + ML Ranker

This is the **project-specified approach (Section 4.2.3)**:
1. Extract candidate sentences using TF-IDF similarity to the correct answer
2. Apply Wh-word templates to generate multiple candidate questions per sentence
3. Train an SVM / Random Forest ranker on hand-built features to pick the best candidate

Pseudo-labels for the ranker are produced by scoring each candidate against the *reference* RACE question with BLEU/ROUGE/METEOR — the candidate with the highest combined quality score per question gets `is_good_question=1`.

**Memory note:** with full data, we cap `top_k_sentences=2` to bound the candidate count.

In [ ]:
QGEN_DIR = f"{RESULTS_DIR}/question_generation_ranker{SUFFIX}"
os.makedirs(QGEN_DIR, exist_ok=True)

# Re-fit a smaller TF-IDF on the question-generation-relevant text only
qgen_tfidf_path = f"{MODEL_A_DIR}/qgen_tfidf_vectorizer{SUFFIX}.pkl"

if os.path.exists(qgen_tfidf_path):
    qgen_tfidf = joblib.load(qgen_tfidf_path)
    print("QGen TF-IDF already exists. Loaded.")
else:
    print("Fitting QGen TF-IDF on training articles + questions + answers...")
    corpus = (
        model_b_train["article_nopunct"].astype(str).tolist() +
        model_b_train["question_nopunct"].astype(str).tolist() +
        model_b_train["correct_answer_nopunct"].astype(str).tolist()
    )
    qgen_tfidf = TfidfVectorizer(
        max_features=20000, ngram_range=(1, 2),
        min_df=5, max_df=0.95
    )
    qgen_tfidf.fit(corpus)
    joblib.dump(qgen_tfidf, qgen_tfidf_path)

print("QGen vocab size:", len(qgen_tfidf.vocabulary_))

QGen TF-IDF already exists. Loaded.
QGen vocab size: 20000


In [ ]:
# Sentence/word helpers (re-defined locally for clarity)
def normalize_text_qgen(t): return re.sub(r"\s+", " ", str(t).lower().strip())
def tokenize_qgen(t):       return normalize_text_qgen(t).split()

def overlap_count(a, b):
    return len(set(tokenize_qgen(a)) & set(tokenize_qgen(b)))
def overlap_ratio(a, b):
    sa, sb = set(tokenize_qgen(a)), set(tokenize_qgen(b))
    return len(sa & sb)/len(sb) if sb else 0.0

def get_top_candidate_sentences(row, vec, top_k=2):
    sentences = parse_sentence_list(row["article_sentences"])
    if not sentences:
        return []
    answer = str(row["correct_answer_nopunct"])
    sent_vecs = vec.transform(sentences)
    ans_vec   = vec.transform([answer])
    sims = np.asarray(sent_vecs.multiply(ans_vec).sum(axis=1)).ravel()
    ranked = []
    for idx, sent in enumerate(sentences):
        contains = int(normalize_text_qgen(answer) in normalize_text_qgen(sent))
        score = float(sims[idx]) + (0.25 * contains)
        ranked.append({
            "source_sentence": sent,
            "sentence_position": idx,
            "sentence_score": score,
            "contains_answer": contains,
        })
    return sorted(ranked, key=lambda x: x["sentence_score"], reverse=True)[:top_k]

print("Sentence selector ready.")

Sentence selector ready.


In [ ]:
# Templates — reduced to 5 high-signal templates to bound candidate count on full data
LOCATION_WORDS = {"city","country","school","home","house","room","park",
                  "market","hospital","station","street","village","town",
                  "china","america","england","london","paris"}
PERSON_WORDS   = {"mr","mrs","miss","doctor","teacher","father","mother",
                  "boy","girl","man","woman"}

def infer_answer_type(answer):
    a = normalize_text_qgen(answer)
    if any(c.isdigit() for c in a):
        return "when"
    if any(w in a for w in LOCATION_WORDS):
        return "where"
    if any(w in a for w in PERSON_WORDS):
        return "who"
    return "what"

def mask_answer_in_sentence(sentence, answer):
    if not str(answer).strip():
        return str(sentence)
    return re.compile(re.escape(str(answer)), re.IGNORECASE).sub("____", str(sentence))

def shorten(text, max_words=24):
    w = str(text).split()
    return " ".join(w[:max_words]) + (" ..." if len(w) > max_words else "")

def generate_template_candidates(row, sent_info):
    answer    = str(row["correct_answer"])
    sentence  = str(sent_info["source_sentence"])
    a_type    = infer_answer_type(answer)
    masked    = shorten(mask_answer_in_sentence(sentence, answer), 24)

    templates = [
        ("cloze_completion",   "what",  f"What completes this statement: {masked}?"),
        ("according_to_true",  "which", "Which of the following is true according to the passage?"),
        ("what_information",   "what",  "What information is given in the passage?"),
    ]
    if a_type == "who":
        templates.append(("who_template", "who", "Who is mentioned in the passage?"))
    if a_type == "where":
        templates.append(("where_template", "where", "Where does the event in the passage take place?"))
    if a_type == "when":
        templates.append(("when_template", "when", "When did the event in the passage happen?"))

    out = []
    for tname, wh, qtext in templates:
        out.append({
            "question_uid": row["question_uid"], "id": row["id"],
            "article": row["article"],
            "correct_answer": row["correct_answer"],
            "reference_question": row["question"],
            "source_sentence": sentence,
            "sentence_position": sent_info["sentence_position"],
            "sentence_score": sent_info["sentence_score"],
            "contains_answer": sent_info["contains_answer"],
            "template_name": tname,
            "wh_type": wh,
            "answer_type": a_type,
            "generated_question": qtext,
        })
    return out

def build_candidate_dataset(df_in, vec, top_k=2):
    out = []
    for _, row in df_in.iterrows():
        for s_info in get_top_candidate_sentences(row, vec, top_k=top_k):
            out.extend(generate_template_candidates(row, s_info))
    return pd.DataFrame(out)

print("Template generator ready.")

Template generator ready.


In [ ]:
qgen_train_cand_path = f"{QGEN_DIR}/qgen_train_candidates.csv"
qgen_val_cand_path   = f"{QGEN_DIR}/qgen_val_candidates.csv"
qgen_test_cand_path  = f"{QGEN_DIR}/qgen_test_candidates.csv"

if all(os.path.exists(p) for p in [qgen_train_cand_path, qgen_val_cand_path, qgen_test_cand_path]):
    print("QGen candidate datasets already exist. Loading...")
    qgen_train_cand = pd.read_csv(qgen_train_cand_path)
    qgen_val_cand   = pd.read_csv(qgen_val_cand_path)
    qgen_test_cand  = pd.read_csv(qgen_test_cand_path)
else:
    print("Building QGen candidate datasets (this takes a while on full data)...")
    print("  train..."); qgen_train_cand = build_candidate_dataset(model_b_train, qgen_tfidf, top_k=2)
    print("  val...");   qgen_val_cand   = build_candidate_dataset(model_b_val,   qgen_tfidf, top_k=2)
    print("  test...");  qgen_test_cand  = build_candidate_dataset(model_b_test,  qgen_tfidf, top_k=2)
    qgen_train_cand.to_csv(qgen_train_cand_path, index=False)
    qgen_val_cand.to_csv(qgen_val_cand_path, index=False)
    qgen_test_cand.to_csv(qgen_test_cand_path, index=False)

print("Train candidates:", qgen_train_cand.shape)
print("Val candidates:  ", qgen_val_cand.shape)
print("Test candidates: ", qgen_test_cand.shape)

QGen candidate datasets already exist. Loading...
Train candidates: (446781, 13)
Val candidates:   (55893, 13)
Test candidates:  (56025, 13)


In [ ]:
# Pseudo-label candidates: best-quality candidate per question becomes the positive
def pseudo_label_candidates(df_in):
    df_out = df_in.copy()
    quality, b1, rL, mt = [], [], [], []
    for _, r in df_out.iterrows():
        scores = compute_generation_metrics(r["reference_question"], r["generated_question"])
        q = (0.30 * scores["bleu_1"] + 0.20 * scores["bleu_2"] +
             0.25 * scores["rougeL_f1"] + 0.25 * scores["meteor"])
        quality.append(q); b1.append(scores["bleu_1"])
        rL.append(scores["rougeL_f1"]); mt.append(scores["meteor"])
    df_out["qgen_quality_score"] = quality
    df_out["qgen_bleu1"]   = b1
    df_out["qgen_rougeL"]  = rL
    df_out["qgen_meteor"]  = mt
    df_out["is_good_question"] = 0
    best_idx = df_out.groupby("question_uid")["qgen_quality_score"].idxmax()
    df_out.loc[best_idx, "is_good_question"] = 1
    return df_out

qgen_train_lab_path = f"{QGEN_DIR}/qgen_train_candidates_labeled.csv"
qgen_val_lab_path   = f"{QGEN_DIR}/qgen_val_candidates_labeled.csv"

if os.path.exists(qgen_train_lab_path) and os.path.exists(qgen_val_lab_path):
    print("Labeled candidates already exist. Loading...")
    qgen_train_lab = pd.read_csv(qgen_train_lab_path)
    qgen_val_lab   = pd.read_csv(qgen_val_lab_path)
else:
    print("Pseudo-labeling candidates with BLEU/ROUGE/METEOR quality (slow on full data)...")
    qgen_train_lab = pseudo_label_candidates(qgen_train_cand)
    qgen_val_lab   = pseudo_label_candidates(qgen_val_cand)
    qgen_train_lab.to_csv(qgen_train_lab_path, index=False)
    qgen_val_lab.to_csv(qgen_val_lab_path, index=False)

print("Train labeled:", qgen_train_lab.shape)
print("Positive rate:", qgen_train_lab["is_good_question"].mean())

Labeled candidates already exist. Loading...
Train labeled: (446781, 18)
Positive rate: 0.1573030187049136


In [ ]:
# Build feature matrix for the ranker
WH_TYPES = ["who", "what", "where", "when", "why", "how", "which"]
ANS_TYPES = ["who", "what", "where", "when"]

def extract_qgen_features(df_in, vec):
    rows = []
    for _, r in df_in.iterrows():
        gen = str(r["generated_question"])
        sent = str(r["source_sentence"])
        ans  = str(r["correct_answer"])
        ctx  = sent + " " + ans

        gen_v  = vec.transform([gen])
        sent_v = vec.transform([sent])
        ans_v  = vec.transform([ans])
        ctx_v  = vec.transform([ctx])

        feat = {
            "sentence_score":      float(r["sentence_score"]),
            "contains_answer":     int(r["contains_answer"]),
            "sentence_position":   int(r["sentence_position"]),
            "generated_q_len":     len(tokenize_qgen(gen)),
            "source_sentence_len": len(tokenize_qgen(sent)),
            "correct_answer_len":  len(tokenize_qgen(ans)),
            "q_sent_overlap":       overlap_count(gen, sent),
            "q_ans_overlap":        overlap_count(gen, ans),
            "q_sent_overlap_ratio": overlap_ratio(gen, sent),
            "q_ans_overlap_ratio":  overlap_ratio(gen, ans),
            "cos_q_sent": float(gen_v.multiply(sent_v).sum()),
            "cos_q_ans":  float(gen_v.multiply(ans_v).sum()),
            "cos_q_ctx":  float(gen_v.multiply(ctx_v).sum()),
        }
        for wh in WH_TYPES:  feat[f"wh_{wh}"]          = int(r["wh_type"] == wh)
        for at in ANS_TYPES: feat[f"answer_type_{at}"] = int(r["answer_type"] == at)
        tname = str(r["template_name"])
        feat["template_is_cloze"]    = int("cloze" in tname or "completion" in tname)
        feat["template_is_according"] = int("according" in tname)
        feat["template_is_generic"]   = int(tname in ["what_information"])
        rows.append(feat)
    return pd.DataFrame(rows)

X_qgen_train_path = f"{QGEN_DIR}/X_qgen_train_features.csv"
X_qgen_val_path   = f"{QGEN_DIR}/X_qgen_val_features.csv"
X_qgen_test_path  = f"{QGEN_DIR}/X_qgen_test_features.csv"

if all(os.path.exists(p) for p in [X_qgen_train_path, X_qgen_val_path, X_qgen_test_path]):
    print("QGen feature matrices already exist. Loading...")
    X_qgen_train = pd.read_csv(X_qgen_train_path)
    X_qgen_val   = pd.read_csv(X_qgen_val_path)
    X_qgen_test  = pd.read_csv(X_qgen_test_path)
else:
    print("Extracting QGen ranker features...")
    print("  train..."); X_qgen_train = extract_qgen_features(qgen_train_lab, qgen_tfidf)
    print("  val...");   X_qgen_val   = extract_qgen_features(qgen_val_lab,   qgen_tfidf)
    print("  test...");  X_qgen_test  = extract_qgen_features(qgen_test_cand, qgen_tfidf)
    X_qgen_train.to_csv(X_qgen_train_path, index=False)
    X_qgen_val.to_csv(X_qgen_val_path, index=False)
    X_qgen_test.to_csv(X_qgen_test_path, index=False)

# Align column order
common_cols = [c for c in X_qgen_train.columns if c in X_qgen_val.columns and c in X_qgen_test.columns]
X_qgen_train = X_qgen_train[common_cols]
X_qgen_val   = X_qgen_val[common_cols]
X_qgen_test  = X_qgen_test[common_cols]

y_qgen_train = qgen_train_lab["is_good_question"].values
print("X_qgen_train:", X_qgen_train.shape, "  positive rate:", y_qgen_train.mean())

QGen feature matrices already exist. Loading...
X_qgen_train: (446781, 27)   positive rate: 0.1573030187049136


In [ ]:
# Train SVM and Random Forest rankers
qgen_svm_path = f"{MODEL_A_DIR}/qgen_linear_svm_ranker{SUFFIX}.pkl"
qgen_rf_path  = f"{MODEL_A_DIR}/qgen_random_forest_ranker{SUFFIX}.pkl"

if os.path.exists(qgen_svm_path):
    qgen_svm = joblib.load(qgen_svm_path)
    print("Loaded QGen SVM ranker.")
else:
    print("Training QGen Linear SVM ranker...")
    qgen_svm = LinearSVC(class_weight="balanced", random_state=42, max_iter=5000, dual=False)
    qgen_svm.fit(X_qgen_train.values, y_qgen_train)
    joblib.dump(qgen_svm, qgen_svm_path)

if os.path.exists(qgen_rf_path):
    qgen_rf = joblib.load(qgen_rf_path)
    print("Loaded QGen RF ranker.")
else:
    print("Training QGen Random Forest ranker...")
    qgen_rf = RandomForestClassifier(n_estimators=200, class_weight="balanced",
                                     random_state=42, n_jobs=-1)
    qgen_rf.fit(X_qgen_train.values, y_qgen_train)
    joblib.dump(qgen_rf, qgen_rf_path)

print("Rankers ready.")

Loaded QGen SVM ranker.
Training QGen Random Forest ranker...
Rankers ready.


In [ ]:
def rank_scores(model, X):
    if hasattr(model, "decision_function"):
        s = model.decision_function(X)
        return s if s.ndim == 1 else s[:, 1]
    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(X)
        idx = list(model.classes_).index(1) if 1 in model.classes_ else 1
        return proba[:, idx]
    return model.predict(X)

def select_best_questions(candidates_df, feature_df, ranker):
    df = candidates_df.copy().reset_index(drop=True)
    df["ranker_score"] = rank_scores(ranker, feature_df.values)
    out = []
    for q_uid, group in df.groupby("question_uid"):
        out.append(group.loc[group["ranker_score"].idxmax()])
    return pd.DataFrame(out).reset_index(drop=True)

def evaluate_selected(df_in, method, split):
    avg, row_metrics = evaluate_generation_dataframe(
        df_in, "reference_question", "generated_question"
    )
    out = pd.concat([df_in.reset_index(drop=True), row_metrics], axis=1)
    return {"method_name": method, "split": split, **avg}, out

# Validation
val_results_qgen = []
for key, ranker in [("qgen_linear_svm_ranker", qgen_svm),
                    ("qgen_random_forest_ranker", qgen_rf)]:
    sel = select_best_questions(qgen_val_lab, X_qgen_val, ranker)
    res, sel_with_m = evaluate_selected(sel, key, "validation")
    sel_with_m.to_csv(f"{QGEN_DIR}/{key}_selected_validation.csv", index=False)
    val_results_qgen.append(res)
    print(f"  {key}: bleu_1={res['bleu_1']:.4f}  rougeL={res['rougeL_f1']:.4f}  meteor={res['meteor']:.4f}")

qgen_val_df = pd.DataFrame(val_results_qgen).sort_values(
    by=["meteor","rougeL_f1","bleu_1"], ascending=False).reset_index(drop=True)
qgen_val_df.to_csv(f"{QGEN_DIR}/qgen_ranker_validation_metrics.csv", index=False)
display(qgen_val_df)
best_qgen_key = qgen_val_df.iloc[0]["method_name"]
print("Best QGen ranker:", best_qgen_key)

  qgen_linear_svm_ranker: bleu_1=0.1769  rougeL=0.2058  meteor=0.1564
  qgen_random_forest_ranker: bleu_1=0.1705  rougeL=0.2052  meteor=0.1536


,method_name,split,bleu_1,bleu_2,bleu_4,rouge1_f1,rouge2_f1,rougeL_f1,meteor
0,qgen_linear_svm_ranker,validation,0.176863,0.098085,0.057868,0.232069,0.089348,0.205798,0.156404
1,qgen_random_forest_ranker,validation,0.170546,0.091167,0.052130,0.230990,0.082694,0.205245,0.153587


Best QGen ranker: qgen_linear_svm_ranker


In [ ]:
# Test set
best_qgen_ranker = qgen_svm if best_qgen_key == "qgen_linear_svm_ranker" else qgen_rf
sel_test = select_best_questions(qgen_test_cand, X_qgen_test, best_qgen_ranker)
qgen_test_res, sel_test_m = evaluate_selected(sel_test, best_qgen_key, "test")
sel_test_m.to_csv(f"{QGEN_DIR}/{best_qgen_key}_selected_test.csv", index=False)
with open(f"{QGEN_DIR}/best_qgen_ranker_test_metrics.json", "w") as f:
    json.dump(qgen_test_res, f, indent=2)
print("QGen test:", json.dumps(qgen_test_res, indent=2))

QGen test: {
  "method_name": "qgen_linear_svm_ranker",
  "split": "test",
  "bleu_1": 0.18107822927668166,
  "bleu_2": 0.1017768780067715,
  "bleu_4": 0.060729100791378506,
  "rouge1_f1": 0.23618464437752323,
  "rouge2_f1": 0.09265487093337056,
  "rougeL_f1": 0.2091166484522142,
  "meteor": 0.15982748442555406
}


In [ ]:
# Compare baseline vs ML-ranked on test
comparison = []
with open(f"{GEN_EVAL_DIR}/template_baseline_summary.json") as f:
    base = json.load(f)["test"]
comparison.append({"method": "template_baseline", "split": "test", **base})
comparison.append({"method": best_qgen_key,      "split": "test",
                   **{k: qgen_test_res[k] for k in ["bleu_1","bleu_2","bleu_4",
                                                    "rouge1_f1","rouge2_f1","rougeL_f1","meteor"]}})
qgen_compare_df = pd.DataFrame(comparison)
qgen_compare_df.to_csv(f"{QGEN_DIR}/qgen_template_vs_ranker_comparison.csv", index=False)
display(qgen_compare_df)

,method,split,bleu_1,bleu_2,bleu_4,rouge1_f1,rouge2_f1,rougeL_f1,meteor
0,template_baseline,test,0.118508,0.047170,0.023834,0.190578,0.040510,0.169995,0.091390
1,qgen_linear_svm_ranker,test,0.181078,0.101777,0.060729,0.236185,0.092655,0.209117,0.159827


## 25. Final Model A summary

A single JSON + CSV that captures everything Model A produced.

In [ ]:
results = load_results()

final_summary = {
    "suffix": SUFFIX,
    "full_mode": FULL_MODE,
    "model_a_verification": {
        "primary_metrics_per_supervisor": ["BLEU", "ROUGE", "METEOR"],
        "validation_metrics_csv": f"{GEN_EVAL_DIR}/model_a_validation_metrics.csv",
        "best_model_test_metrics_json": f"{GEN_EVAL_DIR}/best_answer_generation_test_metrics.json",
    },
    "model_a_unsupervised_kmeans": results.get("kmeans_tfidf"),
    "model_a_question_generation": {
        "baseline_summary": f"{GEN_EVAL_DIR}/template_baseline_summary.json",
        "best_ranker_test_metrics": f"{QGEN_DIR}/best_qgen_ranker_test_metrics.json",
        "comparison_csv": f"{QGEN_DIR}/qgen_template_vs_ranker_comparison.csv",
    },
    "diagnostic_classification_metrics": {
        k: {kk: vv for kk, vv in v.items() if kk in ["model_name","features",
            "diag_accuracy","diag_f1_macro","diag_f1_correct"]}
        for k, v in results.items()
        if isinstance(v, dict) and "diag_accuracy" in v
    },
}

final_path = f"{RESULTS_DIR}/model_a_final_summary{SUFFIX}.json"
with open(final_path, "w") as f:
    json.dump(final_summary, f, indent=2)

print("Saved final Model A summary:", final_path)
print(json.dumps(final_summary, indent=2))

Saved final Model A summary: /content/drive/MyDrive/race_rc_project/results/model_a_final_summary_full.json
{
  "suffix": "_full",
  "full_mode": true,
  "model_a_verification": {
    "primary_metrics_per_supervisor": [
      "BLEU",
      "ROUGE",
      "METEOR"
    ],
    "validation_metrics_csv": "/content/drive/MyDrive/race_rc_project/results/generation_evaluation_full/model_a_validation_metrics.csv",
    "best_model_test_metrics_json": "/content/drive/MyDrive/race_rc_project/results/generation_evaluation_full/best_answer_generation_test_metrics.json"
  },
  "model_a_unsupervised_kmeans": {
    "model_name": "MiniBatch K-Means + TF-IDF",
    "task": "unsupervised_clustering",
    "features": "TF-IDF",
    "n_clusters": 2,
    "purity": 0.75,
    "silhouette_score_sample": 0.006245847408022453,
    "silhouette_sample_size": 5000,
    "training_time_seconds": 0.0
  },
  "model_a_question_generation": {
    "baseline_summary": "/content/drive/MyDrive/race_rc_project/results/generation

In [ ]:
# Final list of files produced for Model A
print("=== Files produced ===\n")
print("Processed data:")
for f in sorted(os.listdir(PROCESSED_DIR)):
    if SUFFIX in f: print("  ", f)
print("\nModel A artifacts:")
for f in sorted(os.listdir(MODEL_A_DIR)):
    if SUFFIX in f: print("  ", f)
print("\nResults:")
for f in sorted(os.listdir(RESULTS_DIR)):
    if SUFFIX in f: print("  ", f)
print("\nGeneration evaluation:")
if os.path.exists(GEN_EVAL_DIR):
    for f in sorted(os.listdir(GEN_EVAL_DIR)):
        print("  ", f)
print("\nQuestion generation ranker:")
if os.path.exists(QGEN_DIR):
    for f in sorted(os.listdir(QGEN_DIR)):
        print("  ", f)

=== Files produced ===

Processed data:
   X_model_a_test_combined_full.npz
   X_model_a_test_tfidf_full.npz
   X_model_a_train_combined_full.npz
   X_model_a_train_tfidf_full.npz
   X_model_a_val_combined_full.npz
   X_model_a_val_tfidf_full.npz
   clean_df_full.csv
   model_a_test_features_full.csv
   model_a_test_full.csv
   model_a_train_features_full.csv
   model_a_train_full.csv
   model_a_val_features_full.csv
   model_a_val_full.csv
   model_b_test_full.csv
   model_b_train_full.csv
   model_b_val_full.csv
   test_clean_full.csv
   train_clean_full.csv
   val_clean_full.csv
   y_model_a_test_full.npy
   y_model_a_train_full.npy
   y_model_a_val_full.npy

Model A artifacts:
   hard_voting_ensemble_tfidf_full.pkl
   kmeans_tfidf_full.pkl
   linear_svm_combined_full.pkl
   linear_svm_tfidf_full.pkl
   logistic_regression_combined_full.pkl
   logistic_regression_tfidf_full.pkl
   naive_bayes_tfidf_full.pkl
   numeric_scaler_full.pkl
   qgen_linear_svm_ranker_full.pkl
   qgen_random